## Lag-Llama without covariates

parquet data, 4 channels, all subjects, 5 separate windows(ctx=512, h=64)

In [1]:
import os

!rm -rf lag-llama

!git clone -b update-gluonts https://github.com/time-series-foundation-models/lag-llama.git

# delet from requirements to avoid binary errors
!sed -i '/numpy/d' lag-llama/requirements.txt
!sed -i '/pandas/d' lag-llama/requirements.txt

!pip install -r lag-llama/requirements.txt pyarrow scikit-learn huggingface_hub -q

from huggingface_hub import hf_hub_download
print("Downloading Lag-Llama weights...")
hf_hub_download(
    repo_id="time-series-foundation-models/Lag-Llama", 
    filename="lag-llama.ckpt", 
    local_dir="./" 
)

print("Setup complete! Safe to proceed to the next cell.")

Cloning into 'lag-llama'...
remote: Enumerating objects: 508, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 508 (delta 155), reused 114 (delta 114), pack-reused 325 (from 3)
Receiving objects: 100% (508/508), 286.89 KiB | 3.73 MiB/s, done.
Resolving deltas: 100% (253/253), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.4/201.4 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that 

lag-llama.ckpt:   0%|          | 0.00/29.5M [00:00<?, ?B/s]

Setup complete! Safe to proceed to the next cell.


## Config

In [2]:
import os
import sys

sys.path.insert(0, os.path.abspath("./lag-llama"))

import torch
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from lag_llama.gluon.estimator import LagLlamaEstimator
from gluonts.dataset.common import ListDataset
from gluonts.evaluation import make_evaluation_predictions

PARQUET_PATH = os.environ.get(
    "EEG_PARQUET", os.path.join(os.getcwd(), "eeg_preprocessed_4ch_raw.parquet")
)
KAGGLE_DATASET_DIR = "/kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw"
PARQUET_FILENAME = "eeg_preprocessed_4ch_raw.parquet"
DATA_PATH = os.path.join(KAGGLE_DATASET_DIR, PARQUET_FILENAME)
PARQUET_PATH = DATA_PATH

LIMIT_PATIENTS = None
CHANNELS = ["Fp1", "Fp2", "P3", "P4"]
CONTEXT_LENGTH = 512
HORIZON_LENGTH = 64
N_WINDOWS = 5
FS = 500 # Native sampling rate
OFFSET_SAMPLES = 3 * FS

df = pd.read_parquet(PARQUET_PATH)
df_eval = df.head(LIMIT_PATIENTS).copy() if LIMIT_PATIENTS else df.copy()
n_total = len(df)
print(
    f"Loaded parquet: {PARQUET_PATH} | rows in file: {n_total}, "
    f"evaluation on (the first) {len(df_eval)} subjects."
)

Loaded parquet: /kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw/eeg_preprocessed_4ch_raw.parquet | rows in file: 88, evaluation on (the first) 88 subjects.


## Model

In [3]:
CSV_OUT = "lag_llama_eeg_micro_eval_results.csv"
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt_path = "lag-llama.ckpt"
print(f"Loading Lag-Llama, device: {device}...")

# --- LAG-LLAMA PATCH ---
import sys
import types

# generating virtual paths
def create_dummy_module(module_path):
    parts = module_path.split('.')
    current = parts[0]
    if current not in sys.modules:
        sys.modules[current] = types.ModuleType(current)
    parent = current
    for part in parts[1:]:
        current = f"{current}.{part}"
        if current not in sys.modules:
            sys.modules[current] = types.ModuleType(current)
        setattr(sys.modules[parent], part, sys.modules[current])
        parent = current
    return sys.modules[module_path]

# creating virtual gluton model
gluonts_module = create_dummy_module('gluonts.torch.modules.loss')

# for PyTorch picke deserialiser
class DistributionLoss:
    def __init__(self, *args, **kwargs): pass
    def __call__(self, *args, **kwargs): return 0.0
    def __getattr__(self, name): return lambda *args, **kwargs: None

class NegativeLogLikelihood:
    def __init__(self, *args, **kwargs): pass
    def __call__(self, *args, **kwargs): return 0.0
    def __getattr__(self, name): return lambda *args, **kwargs: None

# connecting dummy variables to virtual module
gluonts_module.DistributionLoss = DistributionLoss
gluonts_module.NegativeLogLikelihood = NegativeLogLikelihood
# ---------------------------------------------

ckpt = torch.load(ckpt_path, map_location=torch.device(device), weights_only=False)
estimator_args = ckpt["hyper_parameters"]["model_kwargs"]


model = LagLlamaEstimator(
    ckpt_path=ckpt_path,
    prediction_length=HORIZON_LENGTH,
    context_length=CONTEXT_LENGTH,
    
    # dimensions from metadata to avoid "size mismatch" error
    input_size=estimator_args["input_size"],
    n_layer=estimator_args["n_layer"],
    n_embd_per_head=estimator_args["n_embd_per_head"],
    n_head=estimator_args["n_head"],
    time_feat=estimator_args["time_feat"], # important for temporal covariates
    
    scaling="std",
    batch_size=1,
    num_parallel_samples=20,
    device=torch.device(device)
)

predictor = model.create_predictor(model.create_transformation(), model.create_lightning_module())
print("Lag-Llama ready.")

def forecast_windows(model, y_target, target_ch_name, starts, ctx_len, hor_len):
    preds_all = []
    for s in starts:
        ctx_target = y_target[s : s + ctx_len]
        
        # Lag-Llama via GluonTS expects ListDataset
        test_ds = ListDataset([
            {"target": ctx_target, "start": pd.Timestamp("2024-01-01 00:00:00"), "item_id": "EEG"}
        ], freq="S")
        
        # using global 'predictor' object for inference
        forecast_it, _ = make_evaluation_predictions(
            dataset=test_ds,
            predictor=predictor,  
            num_samples=20
        )
        
        forecasts = list(forecast_it)
        # Use median of the generated samples
        preds = np.median(forecasts[0].samples, axis=0)
        preds_all.append(preds)
        
    return np.array(preds_all)

Loading Lag-Llama, device: cuda...
Lag-Llama ready.


## Processing functions

In [4]:
def window_starts(total_len: int, ctx: int, hor: int, n_win: int, offset: int = OFFSET_SAMPLES) -> np.ndarray:
    need = ctx + hor
    available_len = total_len - offset
    if available_len < need:
        raise ValueError(f"Signal too short: available={available_len}, required >= {need}")
    hi = total_len - need
    
    return np.linspace(offset, hi, n_win, dtype=int)

def series_to_numpy(cell) -> np.ndarray:
    if isinstance(cell, (list, np.ndarray)):
        return np.asarray(cell, dtype=np.float64)
    return np.asarray(cell.tolist() if hasattr(cell, "tolist") else cell, dtype=np.float64)

## Evaluation loop

In [5]:
detail_rows = []
done = 0

for _, row in df_eval.iterrows():
    sid = row["subject_id"]
    grp = row["group"] if "group" in row.index and pd.notna(row["group"]) else "Unknown"
    
    for ch in CHANNELS:
        y_target = series_to_numpy(row[ch])
        
        starts = window_starts(len(y_target), CONTEXT_LENGTH, HORIZON_LENGTH, N_WINDOWS)
            
        # --- Unified forecast call ---
        preds = forecast_windows(
            model=model, 
            y_target=y_target, 
            target_ch_name=ch, 
            starts=starts, 
            ctx_len=CONTEXT_LENGTH, 
            hor_len=HORIZON_LENGTH
        )
        
        mses_win = []
        for i, s in enumerate(starts):
            tgt_true = y_target[s + CONTEXT_LENGTH : s + CONTEXT_LENGTH + HORIZON_LENGTH]
            mses_win.append(mean_squared_error(tgt_true, preds[i]))
            
        mse_mean = float(np.mean(mses_win))
        detail_rows.append(
            {
                "record_type": "per_patient_electrode",
                "subject_id": sid,
                "group": grp,
                "electrode": ch,
                "mse": mse_mean,
                "n_windows": N_WINDOWS,
            }
        )
    done += 1
    print(f"Subjects analysed: {done} / {len(df_eval)} (last: {sid}).")

df_detail = pd.DataFrame(detail_rows)

summary_rows = []
for g in sorted(df_detail["group"].dropna().unique()):
    sub = df_detail[(df_detail["record_type"] == "per_patient_electrode") & (df_detail["group"] == g)]
    if sub.empty:
        continue
    summary_rows.append(
        {
            "record_type": "group_all_electrodes",
            "subject_id": "",
            "group": g,
            "electrode": "ALL",
            "mse": float(sub["mse"].mean()),
            "n_windows": N_WINDOWS,
        }
    )
    for ch in CHANNELS:
        sub_ch = sub[sub["electrode"] == ch]
        if sub_ch.empty:
            continue
        summary_rows.append(
            {
                "record_type": "group_per_electrode",
                "subject_id": "",
                "group": g,
                "electrode": ch,
                "mse": float(sub_ch["mse"].mean()),
                "n_windows": N_WINDOWS,
            }
        )

df_out = pd.concat([df_detail, pd.DataFrame(summary_rows)], ignore_index=True)
df_out.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT}")

try:
    from IPython.display import display
    display(df_out.tail(20))
except ImportError:
    print(df_out.tail(20).to_string())

/usr/local/lib/python3.12/dist-packages/gluonts/dataset/common.py:255: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ProcessDataEntry(to_offset(freq), one_dim_target, use_timestamp),
/usr/local/lib/python3.12/dist-packages/gluonts/torch/util.py:205: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return a[idx]
/usr/local/lib/python3.12/dist-packages/gluonts/dataset/common.py:255: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ProcessDataEntry(to_offset(freq), one_dim_target, use_timestamp),
/usr/local/lib/python3.12/dist-packages/gluonts/torch/util.

Subjects analysed: 1 / 88 (last: sub-001).


/usr/local/lib/python3.12/dist-packages/gluonts/dataset/common.py:255: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ProcessDataEntry(to_offset(freq), one_dim_target, use_timestamp),
/usr/local/lib/python3.12/dist-packages/gluonts/torch/util.py:205: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return a[idx]
/usr/local/lib/python3.12/dist-packages/gluonts/dataset/common.py:255: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ProcessDataEntry(to_offset(freq), one_dim_target, use_timestamp),
/usr/local/lib/python3.12/dist-packages/gluonts/torch/util.

Subjects analysed: 2 / 88 (last: sub-002).


/usr/local/lib/python3.12/dist-packages/gluonts/dataset/common.py:255: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ProcessDataEntry(to_offset(freq), one_dim_target, use_timestamp),
/usr/local/lib/python3.12/dist-packages/gluonts/torch/util.py:205: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return a[idx]
/usr/local/lib/python3.12/dist-packages/gluonts/dataset/common.py:255: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ProcessDataEntry(to_offset(freq), one_dim_target, use_timestamp),
/usr/local/lib/python3.12/dist-packages/gluonts/torch/util.

...

/usr/local/lib/python3.12/dist-packages/gluonts/dataset/common.py:255: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ProcessDataEntry(to_offset(freq), one_dim_target, use_timestamp),
/usr/local/lib/python3.12/dist-packages/gluonts/torch/util.py:205: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return a[idx]
/usr/local/lib/python3.12/dist-packages/gluonts/dataset/common.py:255: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ProcessDataEntry(to_offset(freq), one_dim_target, use_timestamp),
/usr/local/lib/python3.12/dist-packages/gluonts/torch/util.

Subjects analysed: 87 / 88 (last: sub-087).


/usr/local/lib/python3.12/dist-packages/gluonts/dataset/common.py:255: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ProcessDataEntry(to_offset(freq), one_dim_target, use_timestamp),
/usr/local/lib/python3.12/dist-packages/gluonts/torch/util.py:205: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  return a[idx]
/usr/local/lib/python3.12/dist-packages/gluonts/dataset/common.py:255: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ProcessDataEntry(to_offset(freq), one_dim_target, use_timestamp),
/usr/local/lib/python3.12/dist-packages/gluonts/torch/util.

Subjects analysed: 88 / 88 (last: sub-088).
Saved: lag_llama_eeg_micro_eval_results.csv


,record_type,subject_id,group,electrode,mse,n_windows
347,per_patient_electrode,sub-087,F,P4,4.105946e-09,5
348,per_patient_electrode,sub-088,F,Fp1,2.367192e-09,5
349,per_patient_electrode,sub-088,F,Fp2,2.598762e-09,5
350,per_patient_electrode,sub-088,F,P3,2.001750e-09,5
351,per_patient_electrode,sub-088,F,P4,2.193544e-09,5
352,group_all_electrodes,,A,ALL,5.369771e-09,5
353,group_per_electrode,,A,Fp1,5.299747e-09,5
354,group_per_electrode,,A,Fp2,5.599799e-09,5
355,group_per_electrode,,A,P3,5.355227e-09,5
356,group_per_electrode,,A,P4,5.224311e-09,5
